In [ ]:
%%configure

{
    "vCores":
    {
        "parameterName": "pipelinecore",
        "defaultValue": 2
    }
}


In [ ]:
# PARAMETERS, set by the run_pipeline data pipeline (the Fabric mode's CI trigger passes all
# three; the 12-hourly schedule passes none). Empty engine = the deploy_config library's
# dbt_target. land=False when CI already landed once for every engine it is about to run.
engine = ""
run_id = ""
land = True

In [ ]:
# ONE ENGINE, THE SAME SCRIPTS THE LOCAL MODE RUNS (.github/scripts/provision.py,
# download_aemo.py, .github/scripts/run_dbt.py), from the copy of the repo deploy.py put in the
# `dbt` lakehouse's Files/dbt. Nothing dbt-shaped lives here: change the scripts, redeploy.
#
# Do not let two runs LAND at once: both write dbt_landing, and the archive log is rewritten
# whole. The pipeline's concurrency is 1, and CI passes land=False after landing once itself.
import os
import shutil
import subprocess
import sys

import notebookutils

vl = notebookutils.variableLibrary.getLibrary("deploy_config")
engine = engine or vl.dbt_target
for k in ("process_limit", "download_limit", "daily_download_limit"):
    os.environ[k] = getattr(vl, k)
ws_id = notebookutils.runtime.context["currentWorkspaceId"]
os.environ["FABRIC_WORKSPACE_ID"] = ws_id

# dbt and DuckDB/Polars fold in memory here: these four run in-process through run_dbt.py.
# dwh and spark are clients of Fabric compute already and build from the runner.
ENGINES = ("duckrun", "iceberg", "ducklake", "polars")
assert engine in ENGINES, f"engine={engine!r}: this notebook runs one of {ENGINES}"

# The ~135 GiB work disk, not the 19 GiB /tmp overlay; run_dbt.py derives DuckDB's spill dir
# from TMPDIR.
scratch = "/home/trusted-service-user/work" if os.path.isdir("/home/trusted-service-user/work") else "/tmp"
os.environ["TMPDIR"] = os.path.join(scratch, "dbt_tmp")
os.makedirs(os.environ["TMPDIR"], exist_ok=True)

# `dbt` is provision.py's DATA_LAKEHOUSE; deploy.py copies the git-tracked repo into its Files/dbt.
# Mounted, then copied to the work disk: dbt's target/ and DuckDB's spill must not land on OneLake.
repo = os.path.join(scratch, "dbt")
lh_id = notebookutils.lakehouse.get("dbt", ws_id).get("id")
notebookutils.fs.mount(
    f"abfss://{ws_id}@onelake.dfs.fabric.microsoft.com/{lh_id}", "/lh", {"fileCacheTimeout": 0}
)
lh = notebookutils.fs.getMountPath("/lh")
shutil.copytree(f"{lh}/Files/dbt", repo, dirs_exist_ok=True)
os.chdir(repo)
# Into this session's environment; dbt runs in subprocesses below, so no restartPython.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", f"requirements/{engine}.txt", "-r", "requirements/tools.txt"], check=True)

In [ ]:
# provision -> land -> build || retry -> fingerprint, each a subprocess of this kernel.
# provision.py emits the engine's config AND, off CI, its tokens (minted by tokens.py from
# notebookutils), as KEY=value lines.
prov = subprocess.run([sys.executable, ".github/scripts/provision.py", engine],
                      cwd=repo, text=True, capture_output=True)
sys.stderr.write(prov.stderr)
if prov.returncode:
    raise RuntimeError(f"provision.py {engine} failed (exit {prov.returncode})")
# KEY=value lines, split once: the DuckLake DSN carries `=` of its own.
os.environ.update(dict(line.split("=", 1) for line in prov.stdout.splitlines() if "=" in line))

if land:
    subprocess.run([sys.executable, "download_aemo.py"], cwd=repo, check=True)

cmd = [sys.executable, ".github/scripts/run_dbt.py", engine]
if run_id:
    # Where CI reads this run's fingerprint back (fabric_run.py): the Fabric mode's parity.
    out = f"{lh}/Files/dbt_runs/{run_id}/{engine}.json"
    os.makedirs(os.path.dirname(out), exist_ok=True)
    cmd += ["--fingerprint-out", out]
rc = subprocess.run(cmd, cwd=repo).returncode
if rc:
    # Raise so the pipeline run goes red instead of silently reporting success.
    raise RuntimeError(f"dbt {engine} failed (exit {rc}); see the log above")